In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split

ROOT = Path.home() / "Documents" / "customer-churn-retention"
df = pd.read_csv(ROOT / "data" / "processed" / "features_segmented.csv")

target = "churn_value"
drop_cols = ["customer_id", "city", "state", "segment", "segment_name",
             "contract_num", "cltv", target]
X = df.drop(columns=drop_cols)
y = df[target]

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42)

print(X_train.shape, X_val.shape, X_test.shape)
print(y_train.mean().round(3), y_val.mean().round(3), y_test.mean().round(3))

(4930, 18) (1056, 18) (1057, 18)
0.265 0.265 0.266


In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, recall_score, precision_score

num_cols = X_train.select_dtypes(include="number").columns.tolist()
cat_cols = X_train.select_dtypes(exclude="number").columns.tolist()

preprocess = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
])

def evaluate(name, model):
    model.fit(X_train, y_train)
    proba = model.predict_proba(X_val)[:, 1]
    pred = (proba >= 0.5).astype(int)
    return {"model": name,
            "ROC-AUC": round(roc_auc_score(y_val, proba), 3),
            "PR-AUC": round(average_precision_score(y_val, proba), 3),
            "Recall": round(recall_score(y_val, pred), 3),
            "Precision": round(precision_score(y_val, pred, zero_division=0), 3)}

results = []
results.append(evaluate("Baseline (dummy)", Pipeline([("prep", preprocess),
                        ("clf", DummyClassifier(strategy="prior"))])))
results.append(evaluate("Logistic Regression", Pipeline([("prep", preprocess),
                        ("clf", LogisticRegression(max_iter=1000, class_weight="balanced"))])))
pd.DataFrame(results)

,model,ROC-AUC,PR-AUC,Recall,Precision
0,Baseline (dummy),0.500,0.265,0.000,0.000
1,Logistic Regression,0.859,0.693,0.789,0.526
